In [2]:
import json

import numpy as np
from tqdm import tqdm
import os
import torch
import transformers
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image
import matplotlib.pyplot as plt
#加载

device = "cuda"


def grounding_smoke(image_path, model, processor):
    image = Image.open(image_path)
    text_labels = [["smoke"]]
    inputs = processor(image, text_labels, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=0.3,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]],
    )
    result = results[0]
    print(result)
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    ax = plt.gca()
    for box, score, label in zip(result["boxes"], result["scores"], result["labels"]):
        x0, y0, x1, y1 = box.tolist()
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="r", linewidth=3))
        ax.text(x0, y0, label, fontsize=15, color="white")
        plt.show()

def grounding_image_folder(model_name):
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name).to(device)
    folder = "./confused_images"
    for filename in os.listdir(folder):
        grounding_smoke(os.path.join(folder, filename), model, processor)

def classification_smoke_folder(model_name, folder):
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name).to(device)
    total = 0
    correct = 0
    for root, dirs, files in tqdm(os.walk(folder), desc="Walking folders", unit="dir"):
        for file in files:
            if file.lower().endswith(".jpg"):
                if len(file.lower().split("_")) <= 1:
                    print("Skipping {}".format(os.path.join(root, file)))
                    continue
                label = file.lower().split("_")[1].replace(".jpg", "")
                is_smoke = label.startswith("+")
                time = int(label[1:])
                if is_smoke or time > 1000:
                    continue
                image_path = os.path.join(root, file)
                try:
                    prediction = classify_smoke(image_path, model, processor)
                except Exception as e:
                    print(e)
                    continue
                total += 1
                if prediction is True:
                    correct += 1
    accuracy = correct / total
    print(f"Accuracy: {accuracy}")

def classify_smoke(image_path, model, processor):
    image = Image.open(image_path)
    text_labels = [["smoke"]]
    inputs = processor(image, text_labels, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=0.3,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]],
    )
    return len(results[0]['labels']) == 0



def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0



# model name
# IDEA-Research/grounding-dino-base
model_id = "IDEA-Research/grounding-dino-base"
# grounding_image_folder(model_id)
folder = "./dataset/rgb"
classification_smoke_folder(model_id, folder)
# processor = AutoProcessor.from_pretrained(model_id)
# model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
# classify_smoke("./dataset/day/20231110.112257-Border36Fire-om-w-mobo-c/1699646403_+02226.jpg", model, processor)

Walking folders: 0dir [00:00, ?dir/s]/home/sora/anaconda3/envs/mllm/lib/python3.9/site-packages/transformers/models/grounding_dino/processing_grounding_dino.py:95: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)
Walking folders: 195dir [09:05,  2.97s/dir]

Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598902278.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598904978.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598904558.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598901018.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598902338.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598905938.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598908818.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598908398.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598900718.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598903058.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598900898.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598901318.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598905578.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598906658.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-n-mobo-c/1598906058.jpg
Skipping ./dataset/rgb/20200831_FIRE_wc-

Walking folders: 386dir [18:03,  2.81s/dir]

Accuracy: 0.675361355706928


In [23]:
from torchvision.ops import box_iou

def smoke_IoU_folder(model_name):
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name).to(device)
    labeled_folder = './dataset/labeled_data/Annotation'
    TP = 0
    FP = 0
    FN = 0
    total_iou = 0.0
    matched_count = 0
    for dir in tqdm(os.listdir(labeled_folder), desc="Walking folders", unit="dir"):
        original_path = os.path.join('./dataset/all', dir)
        label_path = os.path.join(labeled_folder, dir)
        for filename in os.listdir(label_path):
            original_image_name = filename.replace(".json", ".jpg")
            image_path = os.path.join(original_path, original_image_name)
            predict_box = get_box(image_path, model, processor)
            with open(os.path.join(label_path, filename), "r") as f:
                data = json.load(f)
                label_box = torch.tensor(data["det_boxes"]).float().to(device)
                if predict_box.shape[0] == 0:
                    FN += label_box.shape[0]
                    continue
                if label_box.shape[0] == 0:
                    FP += predict_box.shape[0]
                    continue
                iou_matrix = box_iou(predict_box, label_box)
                best_iou_per_pred, matched_gt_idx = iou_matrix.max(dim=1)

                matched_mask = best_iou_per_pred >= 0.5
                total_iou += best_iou_per_pred[matched_mask].sum().item()
                matched_count += matched_mask.sum().item()

                per_image_TP = (best_iou_per_pred >= 0.5).sum().item()
                per_image_FP = (best_iou_per_pred < 0.5).sum().item()
                per_image_FN = label_box.shape[0] - per_image_TP

                TP += per_image_TP
                FP += per_image_FP
                FN += per_image_FN
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    mean_iou = total_iou / (matched_count + 1e-6)
    print("Precision:", precision)
    print("Recall:", recall)
    print(f"Mean IoU (on TP): {mean_iou:.4f}")

def get_box(image_path, model, processor):
    image = Image.open(image_path)
    text_labels = [["smoke"]]
    inputs = processor(image, text_labels, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=0.3,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]],
    )
    return results[0]["boxes"]

model_id = "IDEA-Research/grounding-dino-base"
smoke_IoU_folder(model_id)


Walking folders: 100%|██████████| 145/145 [13:43<00:00,  5.68s/dir]

Precision: 0.4740501614104793
Recall: 0.37831946095917557
Mean IoU (on TP): 0.7191
